# Silver → Gold: dim_calendar

**Propósito:** Leer `dbo.dim_calendar` del lakehouse Silver, seleccionar y renombrar columnas relevantes para reporting, y escribir en Gold de forma idempotente.

**Transformaciones aplicadas:**
- Drop de `complete_date` (texto verbose con formato raro)
- Drop de columnas de días hábiles operativas (`dias_habiles_pasados`, `dias_habiles_restantes`, `total_dias_habiles`) — más propias de Silver
- Drop de `week_order`, `year_week_order` (redundantes con `week` e `year_week`)
- Drop de `order_year_semester` (redundante)
- Columna de auditoría `_gold_load_ts`

**Idempotencia:** `overwrite` + `overwriteSchema=true`.

In [7]:
%run ./config

StatementMeta(, 87f498b3-5870-416e-b1fd-569f26d16ff0, 10, Finished, Available, Finished, True)

Config cargada → Bronze: lakehouse_poc | Silver: silver_lakehouse_poc | Gold: gold_lakehouse_poc


In [8]:


SILVER_TABLE = f"{DEFAULT_SCHEMA}.dim_calendar"
GOLD_TABLE   = f"{DEFAULT_SCHEMA}.dim_calendar"

StatementMeta(, 87f498b3-5870-416e-b1fd-569f26d16ff0, 11, Finished, Available, Finished, False)

In [9]:
from pyspark.sql import functions as F
from datetime import datetime

df_silver = spark.read.table(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")

print(f"Filas leídas desde Silver: {df_silver.count()}")
df_silver.printSchema()

StatementMeta(, 87f498b3-5870-416e-b1fd-569f26d16ff0, 12, Finished, Available, Finished, False)

Filas leídas desde Silver: 4382
root
 |-- date: date (nullable = true)
 |-- complete_date: string (nullable = true)
 |-- year: long (nullable = true)
 |-- month_number: long (nullable = true)
 |-- month: string (nullable = true)
 |-- month_short: string (nullable = true)
 |-- day: long (nullable = true)
 |-- week_day_number: long (nullable = true)
 |-- day_name: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- order_year_month: long (nullable = true)
 |-- quarter: string (nullable = true)
 |-- year_quarter: string (nullable = true)
 |-- order_year_quarter: long (nullable = true)
 |-- semester: string (nullable = true)
 |-- year_semester: string (nullable = true)
 |-- order_year_semester: long (nullable = true)
 |-- first_week_day: date (nullable = true)
 |-- last_week_day: date (nullable = true)
 |-- first_month_day: date (nullable = true)
 |-- last_month_day: date (nullable = true)
 |-- first_day_quarter: date (nullable = true)
 |-- last_day_quarter: date (nulla

In [10]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────
COLS_TO_DROP = [
    "complete_date",           # texto verbose con formato inconsistente
    "week_order",              # redundante con week
    "year_week_order",         # redundante con year_week
    "order_year_semester",     # redundante
    "dias_habiles_pasados",    # métrica operativa, no analítica
    "dias_habiles_restantes",  # métrica operativa, no analítica
    "total_dias_habiles",      # métrica operativa, no analítica
    "_silver_load_ts",         # reemplazada por _gold_load_ts
]

# Drop solo las columnas que existan (robusto ante cambios de schema en Silver)
existing_cols = df_silver.columns
cols_to_drop  = [c for c in COLS_TO_DROP if c in existing_cols]

df_gold = (
    df_silver
    .drop(*cols_to_drop)
    .dropDuplicates(["date"])
    .withColumn("_gold_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

StatementMeta(, 87f498b3-5870-416e-b1fd-569f26d16ff0, 13, Finished, Available, Finished, False)

In [11]:
# ─── VALIDACIÓN ───────────────────────────────────────────────────────────────
row_count  = df_gold.count()
null_dates = df_gold.filter(F.col("date").isNull()).count()
dup_dates = row_count - df_gold.select("date").distinct().count()

print(f"Filas a escribir en Gold  : {row_count}")
print(f"Columnas resultantes      : {len(df_gold.columns)}")
print(f"Filas con date nulo       : {null_dates}")
print(f"Fechas duplicadas         : {dup_dates}")

assert null_dates == 0, "ERROR: hay fechas nulas en dim_calendar Gold"
assert dup_dates  == 0, "ERROR: hay fechas duplicadas en dim_calendar Gold (rompe relación PK)"

df_gold.show(3)

StatementMeta(, 87f498b3-5870-416e-b1fd-569f26d16ff0, 14, Finished, Available, Finished, False)

Filas a escribir en Gold  : 4382
Columnas resultantes      : 38
Filas con date nulo       : 0
+----------+----+------------+-----+-----------+---+---------------+---------+----------+----------------+-------+------------+------------------+----------+---------------+--------------+-------------+---------------+--------------+-----------------+----------------+--------------+-------------+------+-----------+-------------------+--------------+--------------+---------+-----------+----------------+---------------+-----------------+------------------+---------------+------+-----------+--------------------+
|      date|year|month_number|month|month_short|day|week_day_number| day_name|year_month|order_year_month|quarter|year_quarter|order_year_quarter|  semester|  year_semester|first_week_day|last_week_day|first_month_day|last_month_day|first_day_quarter|last_day_quarter|first_day_year|last_day_year|  week|  year_week|working_day/festive|is_working_day|is_weekend_day|   season|day_of_year|day

In [12]:
# ─── ESCRITURA IDEMPOTENTE EN GOLD ────────────────────────────────────────────
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{GOLD_LAKEHOUSE}`.{GOLD_TABLE}")
)

print(f"✓ Tabla {GOLD_LAKEHOUSE}.{GOLD_TABLE} escrita correctamente con {row_count} filas.")

StatementMeta(, 87f498b3-5870-416e-b1fd-569f26d16ff0, 15, Finished, Available, Finished, False)

✓ Tabla gold_lakehouse_poc.dbo.dim_calendar escrita correctamente con 4382 filas.
